# Часть 4 из 8. Эксперименты с CatBoost

**Проект:** прогноз цены квартиры на вторичном рынке Москвы по датасету `Moscow_Real_Estate`.

**В этой части:**
- разбиение 70 / 15 / 15 для всех экспериментов;
- функции потерь RMSE, MAE и Huber, глубина 10, one-hot для категорий;
- подбор гиперпараметров Optuna и ансамбль экспериментов.

**Главный результат:** лучший — базовый CatBoost с RMSE: MAPE 6,73 % на валидации; Huber не сошёлся.

> Код продолжает предыдущие части и использует их переменные. Результаты уже сохранены в блокноте, поэтому его можно читать без запуска. Чтобы пересчитать часть, сначала выполните в этом же ядре части 1–3 по порядку: `%run -i 01_eda.ipynb` и так далее.

← [Часть 3. Первое сравнение моделей: CatBoost, LightGBM, XGBoost](03_boosting_baselines.ipynb) · [Все части](README.md) · [Часть 5. Новые признаки и CatBoost на них](05_feature_engineering.ipynb) →

## Модель CatBoost

In [51]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

# ============================================================
# ДАННЫЕ ДЛЯ CATBOOST (без нормализации, категориальные — строки)
# ============================================================

TARGET = 'price_rub'
df_cb = df_sec_m.copy()

# date-подобные
bad = [c for c in df_cb.columns
       if isinstance(df_cb[c].dtype, pd.PeriodDtype)
       or pd.api.types.is_datetime64_any_dtype(df_cb[c])]
df_cb = df_cb.drop(columns=bad)

# Категориальные — строки для CatBoost
cat_cols = [c for c in ['district', 'okrug', 'metro_station', 'metro_line',
                        'building_type', 'renovation']
            if c in df_cb.columns]
for c in cat_cols:
    df_cb[c] = df_cb[c].astype(str)

X = df_cb.drop(columns=[TARGET])
y = df_cb[TARGET].values
y_log = np.log1p(y)

# Один и тот же split для всех экспериментов
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y_log, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42
)

cat_idx = [X.columns.get_loc(c) for c in cat_cols]
print(f"train: {X_tr.shape}, val: {X_val.shape}, test: {X_test.shape}")

train: (35000, 27), val: (7500, 27), test: (7500, 27)


In [52]:
def evaluate_model(model, X_val, y_val_log, name=""):
    pred_log = model.predict(X_val)
    pred = np.expm1(pred_log)
    true = np.expm1(y_val_log)

    mae  = np.mean(np.abs(pred - true))
    rmse = np.sqrt(np.mean((pred - true) ** 2))
    mape = np.mean(np.abs((pred - true) / true)) * 100
    r2   = 1 - np.sum((true - pred) ** 2) / np.sum((true - true.mean()) ** 2)

    if name:
        print(f"\n=== {name} ===")
        print(f"MAE:  {mae:>12,.0f} руб")
        print(f"RMSE: {rmse:>12,.0f} руб")
        print(f"MAPE: {mape:>12.2f} %")
        print(f"R²:   {r2:>12.4f}")

    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'pred': pred}

In [53]:
# Эксперемент 1
model_1 = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function='RMSE',
    eval_metric='MAPE',
    random_seed=42,
    early_stopping_rounds=200,
    verbose=500,
    cat_features=cat_idx,
)

t0 = time.time()
model_1.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
print(f"Обучение: {time.time() - t0:.1f} сек")

res_1 = evaluate_model(model_1, X_val, y_val, "Эксперимент 1: Baseline")

0:	learn: 0.0293197	test: 0.0290249	best: 0.0290249 (0)	total: 72.4ms	remaining: 6m 2s
500:	learn: 0.0043415	test: 0.0043195	best: 0.0043195 (500)	total: 40.1s	remaining: 6m
1000:	learn: 0.0039637	test: 0.0041845	best: 0.0041845 (1000)	total: 1m 20s	remaining: 5m 21s
1500:	learn: 0.0037253	test: 0.0041533	best: 0.0041533 (1499)	total: 2m 1s	remaining: 4m 42s
2000:	learn: 0.0035188	test: 0.0041436	best: 0.0041433 (1982)	total: 2m 42s	remaining: 4m 3s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.00414005026
bestIteration = 2230

Shrink model to first 2231 iterations.
Обучение: 198.6 сек

=== Эксперимент 1: Baseline ===
MAE:       878,979 руб
RMSE:    1,316,360 руб
MAPE:         6.73 %
R²:         0.9759


In [54]:
# Эксперемент 2
model_2 = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function='MAE',       # ← изменили
    eval_metric='MAPE',
    random_seed=42,
    early_stopping_rounds=200,
    verbose=500,
    cat_features=cat_idx,
)

t0 = time.time()
model_2.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
print(f"Обучение: {time.time() - t0:.1f} сек")

res_2 = evaluate_model(model_2, X_val, y_val, "Эксперимент 2: MAE loss")

0:	learn: 0.0293230	test: 0.0290414	best: 0.0290414 (0)	total: 78.6ms	remaining: 6m 33s
500:	learn: 0.0043043	test: 0.0044290	best: 0.0044290 (500)	total: 41.1s	remaining: 6m 9s
1000:	learn: 0.0038445	test: 0.0042870	best: 0.0042868 (995)	total: 1m 22s	remaining: 5m 28s
1500:	learn: 0.0036032	test: 0.0042596	best: 0.0042594 (1499)	total: 3m 4s	remaining: 7m 10s
2000:	learn: 0.0034348	test: 0.0042453	best: 0.0042452 (1982)	total: 4m 12s	remaining: 6m 18s
2500:	learn: 0.0033060	test: 0.0042391	best: 0.0042388 (2432)	total: 4m 54s	remaining: 4m 53s
3000:	learn: 0.0032036	test: 0.0042338	best: 0.0042338 (3000)	total: 5m 34s	remaining: 3m 42s
3500:	learn: 0.0031255	test: 0.0042293	best: 0.0042291 (3490)	total: 6m 14s	remaining: 2m 40s
4000:	learn: 0.0030510	test: 0.0042251	best: 0.0042250 (3996)	total: 6m 53s	remaining: 1m 43s
4500:	learn: 0.0029866	test: 0.0042219	best: 0.0042219 (4500)	total: 7m 35s	remaining: 50.5s
4999:	learn: 0.0029235	test: 0.0042200	best: 0.0042199 (4988)	total: 8m 1

In [55]:
# Эксперемент 3
model_3 = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function='Huber:delta=0.5',   # ← Huber
    eval_metric='MAPE',
    random_seed=42,
    early_stopping_rounds=200,
    verbose=500,
    cat_features=cat_idx,
)

t0 = time.time()
model_3.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
print(f"Обучение: {time.time() - t0:.1f} сек")

res_3 = evaluate_model(model_3, X_val, y_val, "Эксперимент 3: Huber loss")

0:	learn: 0.8789285	test: 0.8835220	best: 0.8835220 (0)	total: 51.1ms	remaining: 4m 15s
500:	learn: 0.0306235	test: 0.0303891	best: 0.0303891 (500)	total: 40s	remaining: 5m 58s
1000:	learn: 0.0235567	test: 0.0231286	best: 0.0231286 (1000)	total: 1m 19s	remaining: 5m 16s
1500:	learn: 0.0189601	test: 0.0186483	best: 0.0186483 (1499)	total: 1m 58s	remaining: 4m 35s
2000:	learn: 0.0179755	test: 0.0178975	best: 0.0178975 (2000)	total: 3m 55s	remaining: 5m 53s
2500:	learn: 0.0170942	test: 0.0173285	best: 0.0173285 (2500)	total: 4m 35s	remaining: 4m 35s
3000:	learn: 0.0161482	test: 0.0167214	best: 0.0167214 (3000)	total: 5m 14s	remaining: 3m 29s
3500:	learn: 0.0155928	test: 0.0164619	best: 0.0164619 (3500)	total: 5m 52s	remaining: 2m 31s
4000:	learn: 0.0151254	test: 0.0162526	best: 0.0162526 (4000)	total: 6m 31s	remaining: 1m 37s
4500:	learn: 0.0147610	test: 0.0161061	best: 0.0161052 (4486)	total: 7m 10s	remaining: 47.7s
4999:	learn: 0.0144683	test: 0.0160016	best: 0.0160016 (4999)	total: 7m 

In [56]:
# Эксперемент 4
model_4 = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.02,       # ← меньше
    depth=10,                 # ← глубже
    l2_leaf_reg=5.0,          # ← больше регуляризации под глубину
    loss_function='RMSE',
    eval_metric='MAPE',
    random_seed=42,
    early_stopping_rounds=300,
    verbose=500,
    cat_features=cat_idx,
)

t0 = time.time()
model_4.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
print(f"Обучение: {time.time() - t0:.1f} сек")

res_4 = evaluate_model(model_4, X_val, y_val, "Эксперимент 4: depth=10, lr=0.02")

0:	learn: 0.0295360	test: 0.0292469	best: 0.0292469 (0)	total: 113ms	remaining: 18m 52s
500:	learn: 0.0044275	test: 0.0044679	best: 0.0044679 (500)	total: 1m 4s	remaining: 20m 31s
1000:	learn: 0.0039575	test: 0.0042547	best: 0.0042547 (1000)	total: 2m 11s	remaining: 19m 42s
1500:	learn: 0.0036340	test: 0.0042118	best: 0.0042117 (1497)	total: 3m 17s	remaining: 18m 40s
2000:	learn: 0.0033531	test: 0.0041970	best: 0.0041965 (1944)	total: 5m 51s	remaining: 23m 25s


KeyboardInterrupt: 

In [ ]:
# Эксперемент 5
model_5 = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    one_hot_max_size=300,     # ← one-hot для всех категорий ≤300 значений
    loss_function='RMSE',
    eval_metric='MAPE',
    random_seed=42,
    early_stopping_rounds=200,
    verbose=500,
    cat_features=cat_idx,
)

t0 = time.time()
model_5.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
print(f"Обучение: {time.time() - t0:.1f} сек")

res_5 = evaluate_model(model_5, X_val, y_val, "Эксперимент 5: one_hot_max_size=300")

0:	learn: 0.0293731	test: 0.0290830	best: 0.0290830 (0)	total: 6.42ms	remaining: 32.1s
500:	learn: 0.0044483	test: 0.0045856	best: 0.0045856 (500)	total: 2.82s	remaining: 25.3s
1000:	learn: 0.0040369	test: 0.0042973	best: 0.0042973 (1000)	total: 5.47s	remaining: 21.9s
1500:	learn: 0.0037952	test: 0.0042315	best: 0.0042313 (1499)	total: 8.12s	remaining: 18.9s
2000:	learn: 0.0035888	test: 0.0042067	best: 0.0042067 (1999)	total: 10.8s	remaining: 16.2s
2500:	learn: 0.0034014	test: 0.0041955	best: 0.0041954 (2498)	total: 13.4s	remaining: 13.4s
3000:	learn: 0.0032316	test: 0.0041928	best: 0.0041927 (2998)	total: 16.1s	remaining: 10.7s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.004190188792
bestIteration = 3208

Shrink model to first 3209 iterations.
Обучение: 18.9 сек

=== Эксперимент 5: one_hot_max_size=300 ===
MAE:       893,865 руб
RMSE:    1,340,844 руб
MAPE:         6.81 %
R²:         0.9750


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Эксперемент 6 optuna ищут лучшие lr, depth, l2, bagging_temperature, random_strength
def objective(trial):
    params = {
        'iterations': 5000,
        'learning_rate': trial.suggest_float('lr', 0.01, 0.08, log=True),
        'depth': trial.suggest_int('depth', 6, 10),
        'l2_leaf_reg': trial.suggest_float('l2', 1, 15, log=True),
        'bagging_temperature': trial.suggest_float('bag', 0, 1),
        'random_strength': trial.suggest_float('rs', 0.1, 3, log=True),
        'border_count': 128,
        'loss_function': 'RMSE',
        'eval_metric': 'MAPE',
        'random_seed': 42,
        'early_stopping_rounds': 200,
        'verbose': False,
        'cat_features': cat_idx,
        'one_hot_max_size': 300,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    pred = np.expm1(model.predict(X_val))
    true = np.expm1(y_val)
    return np.mean(np.abs((pred - true) / true)) * 100

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest MAPE: {study.best_value:.3f}%")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/30 [00:00<?, ?it/s]


Best MAPE: 6.772%
Best params:
  lr: 0.04019554462028296
  depth: 6
  l2: 5.765053869365343
  bag: 0.15459727654660724
  rs: 1.4593932658135678


In [ ]:
# Эксперемент 7 Ансамбль топ 3 модели
# Собираем предсказания на val в log-пространстве
preds_val = {
    'baseline':  model_1.predict(X_val),
    'mae':       model_2.predict(X_val),
    'huber':     model_3.predict(X_val),
    'deep':      model_4.predict(X_val),
    'onehot':    model_5.predict(X_val),
}

# Простое среднее
pred_avg_log = np.mean(list(preds_val.values()), axis=0)
pred_avg = np.expm1(pred_avg_log)
true = np.expm1(y_val)

mape_avg = np.mean(np.abs((pred_avg - true) / true)) * 100
print(f"Простое среднее: MAPE={mape_avg:.3f}%")

# Взвешенное (веса — по 1/MAPE²)
weights = {k: 1 / (evaluate_model(m, X_val, y_val)['mape'] ** 2)
           for k, m in [('baseline', model_1), ('mae', model_2),
                        ('huber', model_3), ('deep', model_4),
                        ('onehot', model_5)]}
total_w = sum(weights.values())
print("\nВеса:")
for k, w in weights.items():
    print(f"  {k}: {w/total_w:.3f}")

pred_w_log = sum(preds_val[k] * weights[k] for k in preds_val) / total_w
pred_w = np.expm1(pred_w_log)
mape_w = np.mean(np.abs((pred_w - true) / true)) * 100
print(f"\nВзвешенное среднее: MAPE={mape_w:.3f}%")

Простое среднее: MAPE=9.768%

Веса:
  baseline: 0.256
  mae: 0.246
  huber: 0.000
  deep: 0.249
  onehot: 0.249

Взвешенное среднее: MAPE=6.715%


In [ ]:
results = pd.DataFrame({
    'Baseline':      {k: v for k, v in res_1.items() if k != 'pred'},
    'MAE loss':      {k: v for k, v in res_2.items() if k != 'pred'},
    'Huber loss':    {k: v for k, v in res_3.items() if k != 'pred'},
    'depth=10':      {k: v for k, v in res_4.items() if k != 'pred'},
    'one_hot':       {k: v for k, v in res_5.items() if k != 'pred'},
}).T

print(results[['mape', 'r2', 'mae', 'rmse']].round(4))

                   mape            r2           mae          rmse
Baseline         6.7278  9.759000e-01  8.789794e+05  1.316360e+06
MAE loss         6.8549  9.742000e-01  9.020606e+05  1.363507e+06
Huber loss  104507.5139 -6.922013e+08  2.755942e+09  2.232498e+11
depth=10         6.8152  9.746000e-01  8.932550e+05  1.352354e+06
one_hot          6.8119  9.750000e-01  8.938645e+05  1.340844e+06


---
← [Часть 3. Первое сравнение моделей: CatBoost, LightGBM, XGBoost](03_boosting_baselines.ipynb) · [Все части](README.md) · [Часть 5. Новые признаки и CatBoost на них](05_feature_engineering.ipynb) →